# Dataset audit

Produces:
- Session, user, and trial counts
- Sessions-per-user distribution
- PHQ-9 and BDI-II severity distributions (per user, after aggregating multi-session users)
- Trials-per-session distribution
- Scene-duration distribution
- Per-pair scene counts
- Webcam sampling-rate distribution (from raw gaze data)
- Session date range, if timestamps are available

## 1. Setup

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np

VALID_PAIRS = ["negative_vs_positive", "negative_vs_neutral", "neutral_vs_positive"]
DURATION_MIN_MS = 500
DURATION_MAX_MS = 10000

## 2. Load and filter the scene-level data (identical to nb01)

In [0]:
scene_metrics = spark.table("anima.scene_metrics")

df_stimulus = (scene_metrics
    .filter(F.col("scene_type") == "stimulus")
    .filter(F.col("scene_quality_valid") == True)
    .filter(F.col("scene_valence_pair").isin(VALID_PAIRS))
    .filter(F.col("duration_ms").between(DURATION_MIN_MS, DURATION_MAX_MS)))

forms = spark.table("anima.forms").select(
    "session_id", "uid", "phq9_score", "phq9_severity",
    "bdi_score", "bdi_severity", "createdAt"
)

df = df_stimulus.join(forms, on="session_id", how="inner")

print("COUNTS")
print(f"Clean stimulus scenes (trials): {df.count()}")
print(f"Unique sessions: {df.select('session_id').distinct().count()}")
print(f"Unique users: {df.select('uid').distinct().count()}")

## 3. Sessions per user

In [0]:
sessions_per_user = (df.select("uid", "session_id").distinct()
    .groupBy("uid").agg(F.count("session_id").alias("n_sessions"))
    .toPandas())

print("SESSIONS PER USER")
print(sessions_per_user["n_sessions"].describe().round(2).to_string())
print()
print("Distribution of sessions-per-user:")
print(sessions_per_user["n_sessions"].value_counts().sort_index().head(20).to_string())
print()
print(f"Users with 1 session: {(sessions_per_user['n_sessions'] == 1).sum()}")
print(f"Users with 2+ sessions: {(sessions_per_user['n_sessions'] >= 2).sum()}")
print(f"Max sessions per user: {sessions_per_user['n_sessions'].max()}")

## 4. PHQ-9 and BDI-II severity distributions (per user)

For users with multiple sessions, we take the mean score across their sessions before assigning to a severity category, to avoid double-counting.

In [0]:
user_scores = (df.select("uid", "session_id", "phq9_score", "bdi_score").distinct()
    .groupBy("uid")
    .agg(
        F.mean("phq9_score").alias("phq9_score_mean"),
        F.mean("bdi_score").alias("bdi_score_mean"),
    )
    .toPandas())

def phq9_cat(s):
    if s < 5:  return "minimal"
    if s < 10: return "mild"
    if s < 15: return "moderate"
    if s < 20: return "moderately_severe"
    return "severe"

def bdi_cat(s):
    if s < 14: return "minimal"
    if s < 20: return "mild"
    if s < 29: return "moderate"
    return "severe"

user_scores["phq9_severity"] = user_scores["phq9_score_mean"].apply(phq9_cat)
user_scores["bdi_severity"]  = user_scores["bdi_score_mean"].apply(bdi_cat)

print("PHQ-9 SEVERITY (PER USER)")
print(user_scores["phq9_severity"].value_counts()
    .reindex(["minimal","mild","moderate","moderately_severe","severe"]).to_string())
print()
print(f"PHQ-9 score: mean={user_scores['phq9_score_mean'].mean():.2f}, "
      f"median={user_scores['phq9_score_mean'].median():.1f}, "
      f"SD={user_scores['phq9_score_mean'].std():.2f}, "
      f"min={user_scores['phq9_score_mean'].min():.0f}, "
      f"max={user_scores['phq9_score_mean'].max():.0f}")
print()
print("BDI-II SEVERITY (PER USER)")
print(user_scores["bdi_severity"].value_counts()
    .reindex(["minimal","mild","moderate","severe"]).to_string())
print()
print(f"BDI-II score: mean={user_scores['bdi_score_mean'].mean():.2f}, "
      f"median={user_scores['bdi_score_mean'].median():.1f}, "
      f"SD={user_scores['bdi_score_mean'].std():.2f}, "
      f"min={user_scores['bdi_score_mean'].min():.0f}, "
      f"max={user_scores['bdi_score_mean'].max():.0f}")

## 5. Trials per session and per-pair scene counts

In [0]:
trials_per_session = (df.groupBy("session_id")
    .agg(F.count("*").alias("n_trials"))
    .toPandas())

print("TRIALS PER SESSION (after quality filter)")
print(trials_per_session["n_trials"].describe().round(2).to_string())
print()

pair_counts = (df.groupBy("scene_valence_pair")
    .agg(F.count("*").alias("n_scenes"))
    .orderBy("scene_valence_pair")
    .toPandas())
print("SCENES PER PAIR TYPE")
print(pair_counts.to_string(index=False))

## 6. Scene duration distribution

In [0]:
duration = df.select("duration_ms").toPandas()["duration_ms"]
print("SCENE DURATION (ms)")
print(duration.describe(percentiles=[.05, .25, .5, .75, .95]).round(0).to_string())
print()
rounded = (duration / 100).round() * 100
print("Top 5 most common duration bins (100ms):")
print(rounded.value_counts().head().to_string())

## 7. Session creation date range

In [0]:
try:
    dates = df.select("createdAt").toPandas()["createdAt"]
    dates_parsed = pd.to_datetime(dates, errors="coerce")
    print("SESSION DATE RANGE")
    print(f"First: {dates_parsed.min()}")
    print(f"Last:  {dates_parsed.max()}")
    print(f"Span:  {(dates_parsed.max() - dates_parsed.min()).days} days")
except Exception as e:
    print(f"Skipping date range: {e}")

## 8. Webcam sampling rate (from raw gaze data)

Skip this cell if you don't have the raw gaze table or if it's prohibitively expensive to query. If you do run it, it computes median inter-sample timestamp differences per session and summarizes across sessions.

In [0]:
VOLUME_PATH = "/Volumes/workspace/anima/anima_volume/raw/dynamic_ab_research/set/"
SAMPLE_N_SESSIONS = None

from pyspark.sql.window import Window

raw = (spark.read
       .option("header", True)
       .option("inferSchema", True)
       .option("recursiveFileLookup", "true")
       .csv(VOLUME_PATH))

raw = raw.withColumn(
    "session_id",
    F.regexp_extract(F.col("_metadata.file_path"), r"([^/]+)\.csv$", 1)
)

w = Window.partitionBy("session_id").orderBy("TIMESTAMP")
dt = raw.withColumn("dt_ms", F.col("TIMESTAMP") - F.lag("TIMESTAMP").over(w))

per_session = (dt
    .filter(F.col("dt_ms").isNotNull() & (F.col("dt_ms") > 0) & (F.col("dt_ms") < 1000))
    .groupBy("session_id")
    .agg(F.expr("percentile(dt_ms, 0.5)").alias("median_dt_ms"))
    .withColumn("hz", 1000.0 / F.col("median_dt_ms"))
    .toPandas())

print(f"WEBCAM SAMPLING RATE (Hz) ACROSS {len(per_session)} SESSIONS")
print(per_session["hz"].describe(percentiles=[.05, .25, .5, .75, .95]).round(1).to_string())
print()
print("Distribution by bucket:")
buckets = pd.cut(per_session["hz"], bins=[0, 10, 15, 20, 25, 30, 40, 1000], labels=["<10", "10-15", "15-20", "20-25", "25-30", "30-40", "40+"])
print(buckets.value_counts().sort_index().to_string())